In [13]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [14]:
%pip install "transformers==4.53.2" "syncode==0.4.16"

In [15]:
import sys
import torch
import transformers
import syncode

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("SynCode imported successfully")
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Transformers: 4.53.2
SynCode imported successfully
PyTorch: 2.11.0+cpu
GPU available: False


In [16]:
from syncode import Syncode
import torch

arithmetic_grammar = r"""
start: NUMBER OPERATOR NUMBER
NUMBER: /-?[0-9]+(\.[0-9]*)?/
OPERATOR: "+" | "-" | "*" | "/"
%import common.WS
%ignore WS
"""

syn_llm = Syncode(
    model="microsoft/phi-2",
    grammar=arithmetic_grammar,
    parse_output_only=True,
    max_new_tokens=20,
    device="cuda" if torch.cuda.is_available() else "cpu",
    quantize=False,
)

print("Phi-2 with SynCode is ready.")

[2026-10-27 16:37:38,073-root] - Loading model microsoft/phi-2 with device:cpu, device_map:auto, torch_dtype:auto


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi-2 with SynCode is ready.


In [17]:
import re

def run_syncode(textual_query):
    prompt = f"""Extract the two numbers and operator from each query.
Return only an expression in the format: number operator number.

Examples:
Query: What is 12.0 plus 5.0?
Expression: 12.0 + 5.0

Query: What is 9.5 multiplied by 2.0?
Expression: 9.5 * 2.0

Query: What is 18.0 minus 4.0?
Expression: 18.0 - 4.0

Query: What is 20.0 divided by 5.0?
Expression: 20.0 / 5.0

Query: {textual_query}
Expression:"""

    output = syn_llm.infer(prompt)[0].strip()

    match = re.search(
        r'(-?(?:\d+(?:\.\d*)?|\.\d+))\s*([+\-*/])\s*'
        r'(-?(?:\d+(?:\.\d*)?|\.\d+))',
        output
    )

    if not match:
        raise ValueError(f"Could not extract an arithmetic expression: {output!r}")

    a = float(match.group(1))
    operator = match.group(2)
    b = float(match.group(3))

    if operator == "+":
        answer = a + b
    elif operator == "-":
        answer = a - b
    elif operator == "*":
        answer = a * b
    else:
        if b == 0:
            raise ZeroDivisionError("Cannot divide by zero.")
        answer = a / b

    return ([a, b], operator, float(answer))

In [18]:
tests = [
    ("What is 327. multiplied by 11.0?", ([327.0, 11.0], "*", 3597.0)),
    ("What is 45.1 plus 23.54?", ([45.1, 23.54], "+", 68.64)),
    ("What is 120.4 divided by 4.0?", ([120.4, 4.0], "/", 30.1)),
]

for query, expected in tests:
    actual = run_syncode(query)
    print("Query:", query)
    print("Returned:", actual)
    print("Expected:", expected)
    print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query: What is 327. multiplied by 11.0?
Returned: ([327.0, 11.0], '*', 3597.0)
Expected: ([327.0, 11.0], '*', 3597.0)

Query: What is 45.1 plus 23.54?
Returned: ([45.1, 23.54], '+', 68.64)
Expected: ([45.1, 23.54], '+', 68.64)

Query: What is 120.4 divided by 4.0?
Returned: ([120.4, 4.0], '/', 30.1)
Expected: ([120.4, 4.0], '/', 30.1)

